# `macron_morph` demo

Demonstrates the `macron_morph` spaCy component from `latincy-ext`. 
The component reads macronized token forms and uses a kaikki-derived lookup to
set `token._.macron_morph` and `token._.macron_pos_` — non-destructive alternatives
to the model's own morphological predictions.

**Data dependency:** `latin-forms-macronized-morph.json` — built from
`latincy-words/scripts/extract_macronized_morph.py`.

**Pipeline:** `la_core_web_lg` → `macron_morph` (last)

In [ ]:
import spacy
import latincy_ext  # registers macron_morph factory
from pathlib import Path

LOOKUP = Path("../../latincy-words/data/processed/latin-forms-macronized-morph.json")
assert LOOKUP.exists(), f"Build the lookup first: cd latincy-words && uv run python scripts/extract_macronized_morph.py"

nlp = spacy.load("la_core_web_lg")
nlp.add_pipe("macron_morph", last=True, config={"lookup_path": str(LOOKUP)})
print(nlp.pipe_names)

## Helper

In [ ]:
def show(doc):
    header = f"{'TOKEN':<14} {'POS':<7} {'MODEL MORPH':<50} {'MACRON MORPH':<40} {'MACRON POS'}"
    print(header)
    print("-" * len(header))
    for token in doc:
        print(
            f"{token.text:<14} "
            f"{token.pos_:<7} "
            f"{str(token.morph):<50} "
            f"{token._.macron_morph:<40} "
            f"{token._.macron_pos_}"
        )

## Example 1 — ablative of agent

`puellā` is ablative singular. Without the macron the model must infer this from
context; with the macron the lookup is deterministic.

In [ ]:
doc = nlp("A puellā res facta est.")
show(doc)

## Example 2 — venit / vēnit (present vs. perfect)

The canonical ambiguity: `venit` (present, *she comes*) vs. `vēnit` (perfect, *she came*).
Both surface identically without macrons.

In [ ]:
present = nlp("Nunc venit puella.")
perfect = nlp("Heri vēnit puella.")

print("Present (venit):")
show(present)
print()
print("Perfect (vēnit):")
show(perfect)

## Example 3 — partial disambiguation (ambiguous form)

`thēsaurō` is both dative and ablative singular. The lookup cannot resolve `Case`,
but it can confirm `Number=Sing` and `pos=NOUN`.

In [ ]:
doc = nlp("Thēsaurō gaudet.")
show(doc)

## Notes

- `token._.macron_morph` is empty string when no macrons present or form not in lookup — safe to use as a falsy check.
- For ambiguous forms, only features agreed across **all** matching parses are set.
- To use with plain (unmacronized) text, add the `macronizer` component before `macron_morph`:

```python
nlp.add_pipe("macronizer", config={"model_path": "..."})  # from latincy-macronizer
nlp.add_pipe("macron_morph", last=True, config={"lookup_path": "..."})
```